# Imports and loading dataset

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, RocCurveDisplay

from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)


train_df = pd.read_csv('/Users/gabrielaslomiany/PycharmProjects/Not-So-Small-ML/Airline Satisfaction/data/train.csv')
test_df = pd.read_csv('/Users/gabrielaslomiany/PycharmProjects/Not-So-Small-ML/Airline Satisfaction/data/test.csv')

print('train shape:', train_df.shape)
print('test shape:', test_df.shape)
train_df.head()

train shape: (699635, 23)
test shape: (299844, 22)


,id,Age,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,Inflight entertainment,On-board service,Leg room service,Baggage handling,Checkin service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,Gender,Customer Type,Type of Travel,Class,satisfaction
0,0,36,694,4,4,4,3,1,4,1,1,3,4,4,3,1,0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
1,1,56,651,5,5,5,5,5,4,4,5,5,5,5,4,3,0,0.0,Male,Loyal Customer,Business travel,Business,True
2,2,31,1371,3,4,2,5,3,3,3,3,4,3,4,5,3,0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
3,3,10,1616,3,5,3,3,3,3,3,3,4,5,3,4,3,0,0.0,Male,Loyal Customer,Personal Travel,Eco,False
4,4,23,481,3,4,3,4,5,3,5,5,5,4,5,3,5,5,0.0,Female,disloyal Customer,Business travel,Eco,False


# Checking training set

In [2]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 699635 entries, 0 to 699634
Data columns (total 23 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   id                                 699635 non-null  int64  
 1   Age                                699635 non-null  int64  
 2   Flight Distance                    699635 non-null  int64  
 3   Inflight wifi service              699635 non-null  int64  
 4   Departure/Arrival time convenient  699635 non-null  int64  
 5   Ease of Online booking             699635 non-null  int64  
 6   Gate location                      699635 non-null  int64  
 7   Food and drink                     699635 non-null  int64  
 8   Online boarding                    699635 non-null  int64  
 9   Seat comfort                       699635 non-null  int64  
 10  Inflight entertainment             699635 non-null  int64  
 11  On-board service                   699635 non-null

In [3]:
train_df.describe()

,id,Age,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,Inflight entertainment,On-board service,Leg room service,Baggage handling,Checkin service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes
count,699635.00000,699635.000000,699635.000000,699635.000000,699635.00000,699635.00000,699635.000000,699635.000000,699635.000000,699635.000000,699635.000000,699635.000000,699635.000000,699635.000000,699635.000000,699635.000000,699635.000000,699343.000000
mean,349817.00000,39.001928,1353.778702,2.778722,3.15621,2.81324,3.013734,3.265644,3.364156,3.568950,3.453910,3.517404,3.472847,3.805443,3.420291,3.380401,1.176209,1.134280
std,201967.37213,13.800943,954.422566,1.280419,1.47539,1.32955,1.254014,1.300202,1.288538,1.275333,1.308157,1.238474,1.271755,1.079175,1.210604,1.277870,5.982187,5.749304
min,0.00000,7.000000,67.000000,0.000000,0.00000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000
25%,174908.50000,27.000000,631.000000,2.000000,2.00000,2.00000,2.000000,2.000000,2.000000,3.000000,2.000000,3.000000,2.000000,3.000000,3.000000,2.000000,0.000000,0.000000
50%,349817.00000,40.000000,954.000000,3.000000,3.00000,3.00000,3.000000,3.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,0.000000,0.000000
75%,524725.50000,50.000000,1814.000000,4.000000,4.00000,4.00000,4.000000,4.000000,4.000000,5.000000,5.000000,4.000000,5.000000,5.000000,4.000000,4.000000,0.000000,0.000000
max,699634.00000,85.000000,4983.000000,5.000000,5.00000,5.00000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,489.000000,491.000000


### Checking for unique values

In [7]:
print(train_df['Gender'].unique())
print(train_df['Customer Type'].unique())
print(train_df['Type of Travel'].unique())
print(train_df['Class'].unique())

<StringArray>
['Female', 'Male']
Length: 2, dtype: str
<StringArray>
['Loyal Customer', 'disloyal Customer']
Length: 2, dtype: str
<StringArray>
['Personal Travel', 'Business travel']
Length: 2, dtype: str
<StringArray>
['Eco', 'Business', 'Eco Plus']
Length: 3, dtype: str


# EDA

### Missing data

In [8]:
print(train_df.columns)
print()
print(test_df.columns)

Index(['id', 'Age', 'Flight Distance', 'Inflight wifi service',
       'Departure/Arrival time convenient', 'Ease of Online booking',
       'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort',
       'Inflight entertainment', 'On-board service', 'Leg room service',
       'Baggage handling', 'Checkin service', 'Cleanliness',
       'Departure Delay in Minutes', 'Arrival Delay in Minutes', 'Gender',
       'Customer Type', 'Type of Travel', 'Class', 'satisfaction'],
      dtype='str')

Index(['id', 'Age', 'Flight Distance', 'Inflight wifi service',
       'Departure/Arrival time convenient', 'Ease of Online booking',
       'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort',
       'Inflight entertainment', 'On-board service', 'Leg room service',
       'Baggage handling', 'Checkin service', 'Cleanliness',
       'Departure Delay in Minutes', 'Arrival Delay in Minutes', 'Gender',
       'Customer Type', 'Type of Travel', 'Class'],
      dtype='str')


`'satisfaction'` is the target variable

In [9]:
missing_train = train_df.isna().sum()
missing_test = test_df.isna().sum()

missing_df = pd.DataFrame({'training_set': missing_train, 'test_set': missing_test})
missing_df

,training_set,test_set
Age,0,0.0
Arrival Delay in Minutes,292,130.0
Baggage handling,0,0.0
Checkin service,0,0.0
Class,0,0.0
Cleanliness,0,0.0
Customer Type,0,0.0
Departure Delay in Minutes,0,0.0
Departure/Arrival time convenient,0,0.0
Ease of Online booking,0,0.0


In [13]:
train_df['Arrival Delay in Minutes'].info()

<class 'pandas.Series'>
RangeIndex: 699635 entries, 0 to 699634
Series name: Arrival Delay in Minutes
Non-Null Count   Dtype  
--------------   -----  
699343 non-null  float64
dtypes: float64(1)
memory usage: 5.3 MB


In [16]:
col = 'Arrival Delay in Minutes'

n_missing = train_df[col].isna().sum()
n_total = len(train_df)
pct_missing = n_missing / n_total * 100

print(f"Missing: {n_missing} / {n_total} ({pct_missing:.2f}%)")

Missing: 292 / 699635 (0.04%)


Seeing that it is less than 5%, I will just drop missing values.

In [18]:
train_df = train_df.dropna()

In [19]:
train_df.isna().sum()

id                                   0
Age                                  0
Flight Distance                      0
Inflight wifi service                0
Departure/Arrival time convenient    0
Ease of Online booking               0
Gate location                        0
Food and drink                       0
Online boarding                      0
Seat comfort                         0
Inflight entertainment               0
On-board service                     0
Leg room service                     0
Baggage handling                     0
Checkin service                      0
Cleanliness                          0
Departure Delay in Minutes           0
Arrival Delay in Minutes             0
Gender                               0
Customer Type                        0
Type of Travel                       0
Class                                0
satisfaction                         0
dtype: int64

## Handling categorical features

In [20]:
print(train_df.select_dtypes(include=["object", "string"]).dtypes)

Gender            str
Customer Type     str
Type of Travel    str
Class             str
dtype: object


In [24]:
print(train_df['Gender'].unique())
print('----------------------------------------')
print(train_df['Customer Type'].unique())
print('----------------------------------------')
print(train_df['Type of Travel'].unique())
print('----------------------------------------')
print(train_df['Class'].unique())

<StringArray>
['Female', 'Male']
Length: 2, dtype: str
----------------------------------------
<StringArray>
['Loyal Customer', 'disloyal Customer']
Length: 2, dtype: str
----------------------------------------
<StringArray>
['Personal Travel', 'Business travel']
Length: 2, dtype: str
----------------------------------------
<StringArray>
['Eco', 'Business', 'Eco Plus']
Length: 3, dtype: str


`'Gender', 'Customer Type', 'Type of Travel'` --> binary values<br>
`'Class'` ---> ordinal values

In [21]:
# to be continued ...


In [25]:
ordinal_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(categories=['Class'])),
])

# binary_features (academic_work_impact): two categories, imputes missing with the most frequent category then encodes No/Yes as 0/1
binary_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(categories=[['Gender', 'Customer Type', 'Type of Travel']])),
])